In [ ]:
import torch
import numpy as np
from PIL import Image
import cv2
import matplotlib.pyplot as plt
from transformers import ViTImageProcessor, AutoModelForImageClassification
from scipy.ndimage import gaussian_filter, median_filter
import os

class ViTUniversalAdversarialAttack: # Атака методом  FGSM (Fast Gradient Sign Method) и с использованием структурных искажений:
    def __init__(self, model_path, target_class=0, patch_size=4): # Инициализация атаки:
        self.model = AutoModelForImageClassification.from_pretrained(model_path)
        self.processor = ViTImageProcessor.from_pretrained(model_path)
        self.target_class = target_class
        self.patch_size = patch_size
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.model.to(self.device)
        self.model.eval()  #Переводит модель в режим оценки
        # Параметры атаки:
        self.epsilon = 0.01 # Сила возмущения
        self.noise_level = 0.1  # Амплитуда шума
        self.blur_sigma = 0.3 # Размытие

    def predict(self, image_path): # Предсказание:
        image = Image.open(image_path).convert("RGB")
        inputs = self.processor(images=image, return_tensors="pt").to(self.device)
        with torch.no_grad():
            outputs = self.model(**inputs)
        probs = torch.nn.functional.softmax(outputs.logits, dim=-1)
        return outputs.logits.argmax().item(), probs[0].cpu().numpy()

    def _generate_adversarial_pattern(self, image):# Генерация паттерна атаки с учетом реального размера изображения
        original_size = image.size # Сохраняем оригинальный размер
        # Обрабатываем изображение через процессор (будет resize до размера модели):
        inputs = self.processor(images=image, return_tensors="pt").to(self.device)
        pixel_values = inputs["pixel_values"].requires_grad_(True)
        outputs = self.model(pixel_values=pixel_values)
        loss = -outputs.logits[0, self.target_class] # Целевой класс
        loss.backward() # Обратное распространение
        # Получаем возмущение и преобразуем к оригинальному размеру
        perturbation = self.epsilon * pixel_values.grad.data.sign()
        perturbation = perturbation[0].permute(1, 2, 0).cpu().numpy()
        perturbation = cv2.resize(perturbation, original_size)
        return perturbation

    def _apply_structural_attack(self, img_array, importance_map): # Применение структурных искажений и проверка размеров
        h, w, _ = img_array.shape
        attacked = img_array.copy()
        # Размытие важных областей:
        for i in range(0, h, self.patch_size):
            for j in range(0, w, self.patch_size):
                patch_end_i = min(i + self.patch_size, h)
                patch_end_j = min(j + self.patch_size, w)
                patch = attacked[i:patch_end_i, j:patch_end_j]
                if patch.size > 0 and importance_map[i:patch_end_i, j:patch_end_j].mean() > 0.7:
                    attacked[i:patch_end_i, j:patch_end_j] = median_filter(patch, size=3)
        # Добавление стратегического шума:
        if importance_map.shape[0] == h and importance_map.shape[1] == w:
            noise_mask = importance_map > 0.5
            noise = np.random.normal(0, self.noise_level, img_array.shape)
            noise = gaussian_filter(noise, sigma=0.5)  # сглаживаем шум
            attacked = attacked + noise * noise_mask[..., np.newaxis]
        # Локальное затемнение/осветление:
        if importance_map.shape[0] == h and importance_map.shape[1] == w:
            attacked = np.clip(attacked * (1 - 0.05*importance_map[..., np.newaxis]), 0, 1)
        return attacked

    def attack(self, image_path, output_path):# Универсальный метод атаки для любых размеров
        # Загрузка изображения с сохранением оригинального размера:
        image = Image.open(image_path).convert("RGB")
        img_array = np.array(image) / 255.0
        original_size = image.size
        # Генерация FGSM возмущения:
        perturbation = self._generate_adversarial_pattern(image)
        # Применение возмущения + обрезает значения:
        perturbed = np.clip(img_array + perturbation, 0, 1)
        # Создание карты важности:
        grad_map = np.abs(perturbation).mean(axis=2)
        importance_map = (grad_map - grad_map.min()) / (grad_map.max() - grad_map.min() + 1e-8)
        # Применение структурных искажений:
        attacked = self._apply_structural_attack(perturbed, importance_map)
        # Глобальное размытие для попытки сокрытия факта "атаки":
        attacked = gaussian_filter(attacked, sigma = 0.3)
        # Сохранение результата:
        os.makedirs(os.path.dirname(output_path), exist_ok=True)
        Image.fromarray((attacked * 255).astype(np.uint8)).save(output_path, quality=95)
        return output_path
    
    def attack_folder(self, input_folder, output_folder):
        total = 0  # Счётчик обработанных изображений
        for root, dirs, files in os.walk(input_folder):
            for file in files:
                if file.lower().endswith((".jpg", ".jpeg", ".png")):
                    input_path = os.path.join(root, file)
                    # Строим путь назначения:
                    relative_path = os.path.relpath(input_path, input_folder)
                    output_path = os.path.join(output_folder, relative_path)
                    try:
                        os.makedirs(os.path.dirname(output_path), exist_ok=True)
                        self.attack(input_path, output_path)
                        total += 1
                        if total % 50 == 0:
                            print(f" +++ Обработано {total} изображений...")
                    except Exception as e:
                        print(f" !!! Ошибка при атаке {relative_path}: {e}")



# Пример использования:
if __name__ == "__main__":
    attacker = ViTUniversalAdversarialAttack(
        model_path="melanoma_attacks/checkpoint-12670",
        target_class=0,
        patch_size=4
    )
    input_folder = "balanced_Image"
    output_folder = "isic_1000_attacked"
    print("Начало массовой атаки на папку...")
    attacker.attack_folder(input_folder, output_folder)
    print("\nМассовая атака завершена.")